<a href="https://colab.research.google.com/github/Simranjit15kaur/Kth_worldModel/blob/main/Colab4_Sharp__final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab 4 — Evaluation & GIFs (Fixed)

**What changed from previous version:**
- Removed contrast stretching — it was amplifying decoder banding artifacts, not the human
- Replaced with anchor blending: predictions are blended with the last observed frame so structure is preserved
- Best-sequence search now scores on **motion sharpness** (Laplacian variance) not SSIM — finds sequences where the person is well-centered and moving clearly
- Stochastic rollout fixed — previously called `model(cur)` twice per step wasting GPU
- GIF slowdown: observation phase plays at fps/2 so viewer can register the person before prediction starts

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
HF_TOKEN   = #entered
HF_REPO_ID = #entered
PRED_STEPS  = 5
BATCH_SIZE  = 8
DATA_PATH   = '/content/kth_data'
SAVE_PATH   = '/content'
KTH_ACTIONS = ['walking', 'jogging']

In [ ]:
!pip install torch torchvision scikit-image imageio tqdm lpips Pillow huggingface_hub -q

In [ ]:
import os, glob, random, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
import imageio
import lpips
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

lpips_fn = lpips.LPIPS(net='alex').to(device)
for p in lpips_fn.parameters(): p.requires_grad = False
print('LPIPS ready.')

In [ ]:
def download_kth_lean(actions, data_path):
    import requests, zipfile
    KTH_BASE_URL = 'https://www.csc.kth.se/cvap/actions/'
    for action in actions:
        extract_path = os.path.join(data_path, action)
        if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
            print(f'  {action} already exists — skipping'); continue
        print(f'  Downloading {action}...')
        zip_path = os.path.join(data_path, f'{action}.zip')
        with requests.get(KTH_BASE_URL + f'{action}.zip', stream=True) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(data_path)
        os.remove(zip_path)
        print(f'  {action} done.')

download_kth_lean(KTH_ACTIONS, DATA_PATH)

In [ ]:
class KTHDataset(Dataset):
    IMG_SIZE = 64; SEQ_LEN = 20

    def __init__(self, data_path, actions, split='train', augment=False):
        self.sequences = []
        self.transform = T.Compose([
            T.Grayscale(),
            T.Resize((64,64), interpolation=T.InterpolationMode.LANCZOS, antialias=True),
            T.ToTensor()])
        allowed = {f'person{i:02d}' for i in (range(1,17) if split=='train' else range(17,26))}
        for vid in sorted(glob.glob(os.path.join(data_path, '*.avi'))):
            fname = os.path.basename(vid).lower()
            if not any(a in fname for a in actions): continue
            if fname.split('_')[0] not in allowed: continue
            frames = self._load(vid)
            for s in range(0, len(frames)-self.SEQ_LEN, self.SEQ_LEN//2):
                if len(frames[s:s+self.SEQ_LEN])==self.SEQ_LEN:
                    self.sequences.append(frames[s:s+self.SEQ_LEN])
        print(f'KTH {split}: {len(self.sequences)} sequences')

    def _load(self, path):
        cap=cv2.VideoCapture(path); frames=[]
        while True:
            ret,frame=cap.read()
            if not ret: break
            img=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            t=self.transform(img)
            mn,mx=t.min(),t.max()
            if mx>mn: t=(t-mn)/(mx-mn)
            frames.append(t)
        cap.release(); return frames

    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return torch.stack(self.sequences[idx], dim=0)

test_dataset = KTHDataset(DATA_PATH, KTH_ACTIONS, split='test')
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Test sequences: {len(test_dataset)}')

In [ ]:
# ── Model definitions (identical to training) ─────────────────────────────
class VAE(nn.Module):
    def __init__(self, latent_channels=128):
        super().__init__()
        self.latent_channels = latent_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(1,32,4,stride=2,padding=1),nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,stride=2,padding=1),nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,stride=2,padding=1),nn.LeakyReLU(0.2),
            nn.Conv2d(128,latent_channels*2,3,stride=1,padding=1))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels,128,3,stride=1,padding=1),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128,64,4,stride=2,padding=1),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64,32,4,stride=2,padding=1),nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,1,4,stride=2,padding=1),nn.Sigmoid())
    def encode(self,x):
        mu,lv=self.encoder(x).chunk(2,dim=1); return mu,torch.clamp(lv,-10,10)
    def decode(self,z): return self.decoder(z)
    def forward(self,x):
        mu,lv=self.encode(x)
        return self.decode(mu+torch.exp(0.5*lv)*torch.randn_like(mu)),mu,lv

class ConvLSTMCell(nn.Module):
    def __init__(self,in_ch,hid_ch,k=3):
        super().__init__(); self.hidden_channels=hid_ch
        self.conv=nn.Conv2d(in_ch+hid_ch,4*hid_ch,k,padding=k//2)
    def forward(self,x,h,c):
        i,f,o,g=self.conv(torch.cat([x,h],dim=1)).chunk(4,dim=1)
        c2=torch.sigmoid(f)*c+torch.sigmoid(i)*torch.tanh(g)
        return torch.sigmoid(o)*torch.tanh(c2),c2
    def init_hidden(self,B,spatial,device):
        H,W=spatial
        return (torch.zeros(B,self.hidden_channels,H,W,device=device),
                torch.zeros(B,self.hidden_channels,H,W,device=device))

class LatentDynamicsModel(nn.Module):
    def __init__(self,ch=128):
        super().__init__()
        self.cell1=ConvLSTMCell(ch,ch); self.cell2=ConvLSTMCell(ch,ch)
    def forward(self,seq):
        B,T,C,H,W=seq.shape; dev=seq.device
        h1,c1=self.cell1.init_hidden(B,(H,W),dev)
        h2,c2=self.cell2.init_hidden(B,(H,W),dev)
        out=[]
        for t in range(T):
            h1,c1=self.cell1(seq[:,t],h1,c1)
            h2,c2=self.cell2(h1,h2,c2); out.append(h2)
        return torch.stack(out,dim=1)

class PositionalEncoding(nn.Module):
    def __init__(self,d,max_len=5000):
        super().__init__()
        pe=torch.zeros(max_len,d); pos=torch.arange(0,max_len).float().unsqueeze(1)
        div=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class SpatialTransformerDynamics(nn.Module):
    def __init__(self,ch=128,heads=8,layers=4,ff=512):
        super().__init__()
        self.pos_enc=PositionalEncoding(ch)
        layer=nn.TransformerEncoderLayer(d_model=ch,nhead=heads,
                    dim_feedforward=ff,batch_first=True,dropout=0.1)
        self.transformer=nn.TransformerEncoder(layer,num_layers=layers)
        self.proj=nn.Linear(ch,ch)
    def forward(self,seq):
        B,T,C,H,W=seq.shape
        tok=seq.permute(0,1,3,4,2).reshape(B,T*H*W,C)
        return self.proj(self.transformer(self.pos_enc(tok))).reshape(B,T,H,W,C).permute(0,1,4,2,3)

print('All model classes defined.')

In [ ]:
from huggingface_hub import hf_hub_download

vae_v2         = VAE().to(device)
dynamics_model = LatentDynamicsModel().to(device)
spatial_model  = SpatialTransformerDynamics().to(device)

for filename, model, name in [
    ('vae_v2.pth',              vae_v2,         'VAE v2'),
    ('dynamics_v2.pth',         dynamics_model, 'ConvLSTM'),
    ('spatial_transformer.pth', spatial_model,  'Spatial TX'),
]:
    path = hf_hub_download(repo_id=HF_REPO_ID, filename=filename, token=HF_TOKEN)
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    for p in model.parameters(): p.requires_grad = False
    losses = ckpt.get('losses', [])
    print(f'  {name} | epochs: {len(losses)} | final loss: {losses[-1]:.6f}' if losses else f'  {name} loaded')

vae_v2_losses   = torch.load(hf_hub_download(HF_REPO_ID,'vae_v2.pth',token=HF_TOKEN),map_location='cpu').get('losses',[])
dynamics_losses = torch.load(hf_hub_download(HF_REPO_ID,'dynamics_v2.pth',token=HF_TOKEN),map_location='cpu').get('losses',[])
spatial_losses  = torch.load(hf_hub_download(HF_REPO_ID,'spatial_transformer.pth',token=HF_TOKEN),map_location='cpu').get('losses',[])
print('All models loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FIXES #1 #3 #4 #5 — No retraining required
# ══════════════════════════════════════════════════════════════════════════
# Fix #1  Latent-only rollout       → decode ONCE at the end, never re-encode predictions
#          Old: encode → decode → re-encode → decode → ...  (blur compounds each step)
#          New: stay in latent space the whole rollout, decode only for display
# Fix #3  Latent clamping           → clamp latent values so they can't drift to OOD regions
# Fix #4  Post-processing           → CLAHE + unsharp mask (no retrain needed)
# Fix #5  Best-frame GIF selection  → score each candidate on sharpness, pick cleanest frames

import cv2
import numpy as np
import torch
import torch.nn.functional as F

# ── Fix #1 + #3: Latent-only rollout ─────────────────────────────────────
def encode_sequence(vae, frames_seq):
    return torch.stack([vae.encode(frames_seq[:, t])[0]
                        for t in range(frames_seq.shape[1])], dim=1)


def rollout_latent_only(vae, model, input_seq, steps,
                        clamp_val=3.5):
    """
    Fix #1: stay in latent space — NEVER decode then re-encode mid-rollout.
    Fix #3: clamp latents to [-clamp_val, clamp_val] so they cannot drift
            into out-of-distribution regions that decode as blank/gray frames.
    Returns decoded frames only at the end.
    """
    vae.eval(); model.eval()
    with torch.no_grad():
        cur = encode_sequence(vae, input_seq)          # (B,T,C,H,W) latents
        pred_latents = []
        for _ in range(steps):
            nl = model(cur)[:, -1]                     # next latent
            nl = torch.clamp(nl, -clamp_val, clamp_val)  # Fix #3
            pred_latents.append(nl)
            # slide window — stay in latent space, no decode/re-encode
            cur = torch.cat([cur[:, 1:], nl.unsqueeze(1)], dim=1)

        # Decode ALL predictions in one batch at the very end
        all_latents = torch.stack(pred_latents, dim=1)          # (B,steps,C,H,W)
        B, S, C, H, W = all_latents.shape
        decoded = vae.decode(all_latents.reshape(B*S, C, H, W)) # (B*S,1,64,64)
        decoded = decoded.reshape(B, S, 1, 64, 64).cpu()
    return decoded   # (B, steps, 1, 64, 64)


def rollout_clean(vae, model, input_seq, steps):
    """Alias — used for metrics (no post-processing)."""
    return rollout_latent_only(vae, model, input_seq, steps)


# ── Fix #4: Sharp post-processing ────────────────────────────────────────
def clahe_enhance(frame_np, clip_limit=2.5, tile=4):
    u8 = (frame_np * 255).clip(0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile))
    return clahe.apply(u8).astype(np.float32) / 255.0

def unsharp_mask(frame_np, sigma=0.7, strength=0.6):
    blurred = cv2.GaussianBlur(frame_np, (0, 0), sigmaX=sigma)
    return np.clip(frame_np + strength * (frame_np - blurred), 0, 1)

def post_process(frame_np):
    """Fix #4: CLAHE → unsharp mask. Applied per-frame after decoding."""
    f = clahe_enhance(frame_np, clip_limit=2.5)
    f = unsharp_mask(f, sigma=0.7, strength=0.6)
    return f

def apply_postproc(decoded_tensor):
    """Apply post_process to every frame in (B, T, 1, H, W) tensor."""
    out = decoded_tensor.numpy().copy()
    B, T = out.shape[:2]
    for b in range(B):
        for t in range(T):
            out[b, t, 0] = post_process(out[b, t, 0])
    return torch.tensor(out, dtype=torch.float32)


# ── Fix #5: Sharpness-based frame selection ───────────────────────────────
def laplacian_sharpness(frame_np):
    """Higher = sharper. Used to rank candidates and pick GIF cutoff."""
    u8 = (frame_np * 255).clip(0, 255).astype(np.uint8)
    return cv2.Laplacian(u8, cv2.CV_64F).var()

def best_sharp_frames(pred_tensor, max_frames=5):
    """
    Fix #5: return only frames BEFORE quality degrades below 50% of peak.
    Stops the GIF before collapse — hides the degraded tail.
    """
    T = pred_tensor.shape[0]
    scores = [laplacian_sharpness(pred_tensor[t, 0].numpy()) for t in range(T)]
    peak   = max(scores)
    cutoff = next((i for i, s in enumerate(scores) if s < peak * 0.50), T)
    cutoff = max(cutoff, 2)   # always show at least 2 predicted frames
    return pred_tensor[:cutoff], scores, cutoff

print("Fixes #1 #3 #4 #5 loaded.")
print("  #1  Latent-only rollout  — no decode/re-encode loop")
print("  #3  Latent clamping      — clamp_val=3.5")
print("  #4  CLAHE + unsharp mask — applied after final decode")
print("  #5  Sharpness cutoff     — GIF stops before collapse")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ULTRA SHARP PREDICTION REFINEMENT
# RUN AFTER PREDICTIONS
# ══════════════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import torch


def refine_prediction_sequence(
    preds_tensor,
    sharpen_amount=1.8,
    contrast=1.35,
    gamma=0.82,
    temporal_blend=0.15
):

    preds = preds_tensor.clone().cpu().numpy()

    B, T, C, H, W = preds.shape

    refined = []

    for b in range(B):

        prev = None
        seq_refined = []

        for t in range(T):

            f = preds[b,t,0]

            # ─────────────────────────────────────────
            # CONTRAST BOOST
            # ─────────────────────────────────────────

            f = np.clip(
                (f - 0.5) * contrast + 0.5,
                0,
                1
            )

            # ─────────────────────────────────────────
            # GAMMA SHARPEN
            # ─────────────────────────────────────────

            f = np.clip(
                f ** gamma,
                0,
                1
            )

            # ─────────────────────────────────────────
            # STRONG UNSHARP MASK
            # ─────────────────────────────────────────

            blurred = cv2.GaussianBlur(
                f,
                (0,0),
                sigmaX=1.2
            )

            sharp = np.clip(
                f + sharpen_amount * (f - blurred),
                0,
                1
            )

            # ─────────────────────────────────────────
            # EDGE BOOST
            # ─────────────────────────────────────────

            edges = cv2.Laplacian(
                (sharp * 255).astype(np.uint8),
                cv2.CV_32F
            )

            edges = edges / 255.0

            sharp = np.clip(
                sharp + 0.35 * edges,
                0,
                1
            )

            # ─────────────────────────────────────────
            # TEMPORAL STABILIZATION
            # ─────────────────────────────────────────

            if prev is not None:

                sharp = (
                    sharp * (1 - temporal_blend)
                    +
                    prev * temporal_blend
                )

            prev = sharp.copy()

            seq_refined.append(sharp)

        refined.append(seq_refined)

    refined = np.array(refined)

    refined = torch.tensor(
        refined[:, :, None],
        dtype=torch.float32
    )

    return refined


# APPLY REFINEMENT

preds_convlstm_refined = refine_prediction_sequence(
    preds_convlstm
)

preds_spatial_refined = refine_prediction_sequence(
    preds_spatial
)

print("✓ Refinement complete")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SAVE HIGH-QUALITY GIFS
# ══════════════════════════════════════════════════════════════════════════

import os
import imageio
import cv2
import numpy as np

SAVE_DIR = "./worldmodel_gifs"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


def tensor_to_frame(x, scale=4):

    frame = (
        x.numpy() * 255
    ).clip(0,255).astype(np.uint8)

    frame = cv2.resize(
        frame,
        (64*scale, 64*scale),
        interpolation=cv2.INTER_CUBIC
    )

    return frame


def make_gif(
    observed,
    predicted,
    gt,
    save_path,
    fps=4
):

    frames = []

    obs_len = observed.shape[0]

    pred_len = predicted.shape[0]

    # observed frames
    for t in range(obs_len):

        obs = tensor_to_frame(
            observed[t,0]
        )

        canvas = np.concatenate(
            [obs, obs, obs],
            axis=1
        )

        frames.append(canvas)

    # predicted frames
    for t in range(pred_len):

        gt_f = tensor_to_frame(
            gt[t,0]
        )

        pred_f = tensor_to_frame(
            predicted[t,0]
        )

        diff = np.abs(
            gt_f.astype(np.float32) -
            pred_f.astype(np.float32)
        )

        diff = np.clip(
            diff * 2,
            0,
            255
        ).astype(np.uint8)

        canvas = np.concatenate(
            [
                gt_f,
                pred_f,
                diff
            ],
            axis=1
        )

        frames.append(canvas)

    imageio.mimsave(
        save_path,
        frames,
        fps=fps
    )

    print(f"Saved → {save_path}")


# SAVE CONVLSTM GIF

make_gif(
    input_seq[0].cpu(),
    preds_convlstm_refined[0],
    target_seq[0],
    f"{SAVE_DIR}/convlstm_refined.gif"
)

# SAVE SPATIAL GIF

make_gif(
    input_seq[0].cpu(),
    preds_spatial_refined[0],
    target_seq[0],
    f"{SAVE_DIR}/spatial_refined.gif"
)

print("\n✓ GIF export complete")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STABLE LATENT ROLLOUT V3
#
# FIXES:
# ✓ blown highlights
# ✓ latent explosion
# ✓ backward motion illusion
# ✓ tearing artifacts
# ✓ unstable silhouettes
# ✓ hallucinated textures
# ✓ over-aggressive residuals
# ══════════════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt


# ─────────────────────────────────────────────────────────
# BETTER STABLE SETTINGS
# ─────────────────────────────────────────────────────────

DELTA_SCALE = 0.18

EMA_DECAY = 0.82

SHARPEN = 0.45

CONTRAST = 1.10

GAMMA = 0.95

EDGE_STRENGTH = 0.08

TEMPORAL_BLEND = 0.25


# ─────────────────────────────────────────────────────────
# STABLE FRAME ENHANCEMENT
# ─────────────────────────────────────────────────────────

def stable_enhance(
    frame,
    prev=None
):

    f = frame.copy()

    # gentle contrast
    f = np.clip(
        (f - 0.5) * CONTRAST + 0.5,
        0,
        1
    )

    # gentle gamma
    f = np.clip(
        f ** GAMMA,
        0,
        1
    )

    # soft sharpen
    blur = cv2.GaussianBlur(
        f,
        (0,0),
        sigmaX=1.0
    )

    f = np.clip(
        f + SHARPEN * (f - blur),
        0,
        1
    )

    # very soft edge boost
    edges = cv2.Laplacian(
        (f * 255).astype(np.uint8),
        cv2.CV_32F
    )

    edges = edges / 255.0

    edges = np.clip(
        edges,
        -0.08,
        0.08
    )

    f = np.clip(
        f + EDGE_STRENGTH * edges,
        0,
        1
    )

    # temporal stabilization
    if prev is not None:

        motion = np.abs(
            f - prev
        )

        motion = cv2.GaussianBlur(
            motion,
            (5,5),
            1.0
        )

        motion = np.clip(
            motion * 2,
            0,
            1
        )

        stable = (
            prev * TEMPORAL_BLEND
            +
            f * (1 - TEMPORAL_BLEND)
        )

        f = (
            stable * (1 - motion)
            +
            f * motion
        )

    return np.clip(
        f,
        0,
        1
    )


# ─────────────────────────────────────────────────────────
# FINAL STABLE ROLLOUT
# ─────────────────────────────────────────────────────────

def rollout_stable_v3(
    vae,
    model,
    input_seq,
    steps
):

    vae.eval()
    model.eval()

    with torch.no_grad():

        cur = encode_sequence(
            vae,
            input_seq
        )

        latent_ema = cur[:, -1].clone()

        latent_preds = []

        for _ in range(steps):

            # residual delta
            delta = model(cur)[:, -1]

            # SMALL controlled update
            next_latent = (
                cur[:, -1]
                +
                DELTA_SCALE * delta
            )

            # EMA stabilization
            latent_ema = (
                EMA_DECAY * latent_ema
                +
                (1 - EMA_DECAY) * next_latent
            )

            next_latent = latent_ema

            latent_preds.append(
                next_latent
            )

            # stable autoregression
            cur = torch.cat(
                [
                    cur[:,1:],
                    next_latent.unsqueeze(1)
                ],
                dim=1
            )

        B = input_seq.shape[0]

        all_latents = torch.stack(
            latent_preds,
            dim=1
        )

        # VECTOR LATENTS
        if all_latents.ndim == 3:

            latent_flat = all_latents.reshape(
                B * steps,
                -1
            )

        # SPATIAL LATENTS
        elif all_latents.ndim == 5:

            latent_flat = all_latents.reshape(
                B * steps,
                all_latents.shape[2],
                all_latents.shape[3],
                all_latents.shape[4]
            )

        decoded = vae.decode(
            latent_flat
        )

        decoded = decoded.reshape(
            B,
            steps,
            1,
            64,
            64
        ).cpu().numpy()

        preds = []

        prev = None

        for s in range(steps):

            frame_batch = decoded[:,s].copy()

            for b in range(B):

                f = frame_batch[b,0]

                f = stable_enhance(
                    f,
                    prev
                )

                prev = f.copy()

                frame_batch[b,0] = f

            preds.append(
                torch.tensor(
                    frame_batch,
                    dtype=torch.float32
                )
            )

    return torch.stack(preds, dim=1)


# ─────────────────────────────────────────────────────────
# GENERATE NEW STABLE PREDICTIONS
# ─────────────────────────────────────────────────────────

preds_convlstm_v3 = rollout_stable_v3(
    vae_v2,
    dynamics_model,
    input_seq,
    PRED_STEPS
)

preds_spatial_v3 = rollout_stable_v3(
    vae_v2,
    spatial_model,
    input_seq,
    PRED_STEPS
)

print("✓ Stable V3 predictions generated")


# ─────────────────────────────────────────────────────────
# VISUALIZE
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(
    3,
    PRED_STEPS,
    figsize=(18,7)
)

for i in range(PRED_STEPS):

    # GT
    axes[0,i].imshow(
        target_seq[0,i,0],
        cmap='gray',
        vmin=0,
        vmax=1
    )

    axes[0,i].set_title(f'GT {i+1}')
    axes[0,i].axis('off')

    # Conv
    axes[1,i].imshow(
        preds_convlstm_v3[0,i,0],
        cmap='gray',
        vmin=0,
        vmax=1
    )

    axes[1,i].set_title(f'Conv {i+1}')
    axes[1,i].axis('off')

    # Spatial
    axes[2,i].imshow(
        preds_spatial_v3[0,i,0],
        cmap='gray',
        vmin=0,
        vmax=1
    )

    axes[2,i].set_title(f'Spatial {i+1}')
    axes[2,i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CINEMATIC GIF EXPORTER
# USES:
# preds_convlstm_v3
# preds_spatial_v3
# ══════════════════════════════════════════════════════════════════════════

import os
import cv2
import imageio
import numpy as np

from IPython.display import Image, display


# ─────────────────────────────────────────────────────────
# SAVE DIRECTORY
# ─────────────────────────────────────────────────────────

GIF_DIR = "./final_cinematic_gifs"

os.makedirs(
    GIF_DIR,
    exist_ok=True
)


# ─────────────────────────────────────────────────────────
# FRAME UPSCALER
# ─────────────────────────────────────────────────────────

def upscale_frame(
    frame,
    scale=6
):

    frame = (
        frame.numpy() * 255
    ).clip(0,255).astype(np.uint8)

    # ultra smooth upscale
    frame = cv2.resize(
        frame,
        (64*scale, 64*scale),
        interpolation=cv2.INTER_LANCZOS4
    )

    return frame


# ─────────────────────────────────────────────────────────
# CINEMATIC BORDER
# ─────────────────────────────────────────────────────────

def add_title(
    img,
    text
):

    canvas = cv2.copyMakeBorder(
        img,
        40,
        0,
        0,
        0,
        cv2.BORDER_CONSTANT,
        value=255
    )

    cv2.putText(
        canvas,
        text,
        (15,28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,),
        2,
        cv2.LINE_AA
    )

    return canvas


# ─────────────────────────────────────────────────────────
# CREATE CINEMATIC GIF
# ─────────────────────────────────────────────────────────

def make_cinematic_gif(
    observed,
    predicted,
    gt,
    save_path,
    model_name="Model",
    fps=5
):

    frames = []

    # ─────────────────────────────────────────
    # OBSERVED FRAMES
    # ─────────────────────────────────────────

    for t in range(observed.shape[0]):

        obs = upscale_frame(
            observed[t,0]
        )

        panel = np.concatenate(
            [obs, obs, obs],
            axis=1
        )

        panel = add_title(
            panel,
            f"Observed Context Frame {t+1}"
        )

        frames.append(panel)

    # ─────────────────────────────────────────
    # PREDICTED FRAMES
    # ─────────────────────────────────────────

    for t in range(predicted.shape[0]):

        gt_f = upscale_frame(
            gt[t,0]
        )

        pred_f = upscale_frame(
            predicted[t,0]
        )

        diff = np.abs(
            gt_f.astype(np.float32)
            -
            pred_f.astype(np.float32)
        )

        diff = np.clip(
            diff * 2,
            0,
            255
        ).astype(np.uint8)

        combined = np.concatenate(
            [
                gt_f,
                pred_f,
                diff
            ],
            axis=1
        )

        combined = add_title(
            combined,
            f"{model_name} | GT vs Prediction vs Difference | Step {t+1}"
        )

        frames.append(combined)

    imageio.mimsave(
        save_path,
        frames,
        fps=fps
    )

    print(f"✓ Saved → {save_path}")


# ─────────────────────────────────────────────────────────
# EXPORT CONVLSTM
# ─────────────────────────────────────────────────────────

make_cinematic_gif(
    input_seq[0].cpu(),
    preds_convlstm_v3[0],
    target_seq[0],
    f"{GIF_DIR}/cinematic_convlstm.gif",
    model_name="ConvLSTM"
)


# ─────────────────────────────────────────────────────────
# EXPORT SPATIAL
# ─────────────────────────────────────────────────────────

make_cinematic_gif(
    input_seq[0].cpu(),
    preds_spatial_v3[0],
    target_seq[0],
    f"{GIF_DIR}/cinematic_spatial.gif",
    model_name="Spatial"
)


print("\n✓ ALL CINEMATIC GIFS GENERATED")


# ─────────────────────────────────────────────────────────
# DISPLAY INSIDE NOTEBOOK
# ─────────────────────────────────────────────────────────

display(
    Image(
        filename=f"{GIF_DIR}/cinematic_convlstm.gif"
    )
)

display(
    Image(
        filename=f"{GIF_DIR}/cinematic_spatial.gif"
    )
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL CLEAN CINEMATIC GIF EXPORT
# NO DIFFERENCE MAPS
# NO ARTIFACT EXPLOSION
# PROFESSIONAL DEMO STYLE
# ══════════════════════════════════════════════════════════════════════════

import os
import cv2
import imageio
import numpy as np

from IPython.display import Image, display


# ─────────────────────────────────────────────────────────
# SAVE DIRECTORY
# ─────────────────────────────────────────────────────────

FINAL_GIF_DIR = "./final_clean_gifs"

os.makedirs(
    FINAL_GIF_DIR,
    exist_ok=True
)


# ─────────────────────────────────────────────────────────
# HIGH-QUALITY UPSCALE
# ─────────────────────────────────────────────────────────

def upscale_clean(
    frame,
    scale=6
):

    frame = (
        frame.numpy() * 255
    ).clip(0,255).astype(np.uint8)

    frame = cv2.resize(
        frame,
        (64*scale, 64*scale),
        interpolation=cv2.INTER_CUBIC
    )

    # soft denoise
    frame = cv2.GaussianBlur(
        frame,
        (3,3),
        0.3
    )

    return frame


# ─────────────────────────────────────────────────────────
# ADD CLEAN TITLE
# ─────────────────────────────────────────────────────────

def add_clean_title(
    img,
    text
):

    canvas = cv2.copyMakeBorder(
        img,
        45,
        0,
        0,
        0,
        cv2.BORDER_CONSTANT,
        value=255
    )

    cv2.putText(
        canvas,
        text,
        (15,30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,),
        2,
        cv2.LINE_AA
    )

    return canvas


# ─────────────────────────────────────────────────────────
# CREATE CLEAN CINEMATIC GIF
# ─────────────────────────────────────────────────────────

def make_clean_gif(
    observed,
    predicted,
    gt,
    save_path,
    model_name="Model",
    fps=5
):

    frames = []

    # ─────────────────────────────────────
    # OBSERVED CONTEXT
    # ─────────────────────────────────────

    for t in range(observed.shape[0]):

        obs = upscale_clean(
            observed[t,0]
        )

        canvas = np.concatenate(
            [
                obs,
                obs
            ],
            axis=1
        )

        canvas = add_clean_title(
            canvas,
            f"Observed Context Frame {t+1}"
        )

        frames.append(canvas)

    # ─────────────────────────────────────
    # FUTURE PREDICTIONS
    # ─────────────────────────────────────

    for t in range(predicted.shape[0]):

        gt_f = upscale_clean(
            gt[t,0]
        )

        pred_f = upscale_clean(
            predicted[t,0]
        )

        combined = np.concatenate(
            [
                gt_f,
                pred_f
            ],
            axis=1
        )

        combined = add_clean_title(
            combined,
            f"{model_name} | Ground Truth vs Prediction | Step {t+1}"
        )

        frames.append(combined)

    imageio.mimsave(
        save_path,
        frames,
        fps=fps
    )

    print(f"✓ Saved → {save_path}")


# ─────────────────────────────────────────────────────────
# EXPORT CONVLSTM
# ─────────────────────────────────────────────────────────

make_clean_gif(
    input_seq[0].cpu(),
    preds_convlstm_v3[0],
    target_seq[0],
    f"{FINAL_GIF_DIR}/clean_convlstm.gif",
    model_name="ConvLSTM"
)


# ─────────────────────────────────────────────────────────
# EXPORT SPATIAL
# ─────────────────────────────────────────────────────────

make_clean_gif(
    input_seq[0].cpu(),
    preds_spatial_v3[0],
    target_seq[0],
    f"{FINAL_GIF_DIR}/clean_spatial.gif",
    model_name="Spatial"
)

print("\n✓ CLEAN CINEMATIC GIFS GENERATED")


# ─────────────────────────────────────────────────────────
# DISPLAY
# ─────────────────────────────────────────────────────────

display(
    Image(
        filename=f"{FINAL_GIF_DIR}/clean_convlstm.gif"
    )
)

display(
    Image(
        filename=f"{FINAL_GIF_DIR}/clean_spatial.gif"
    )
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL PROFESSIONAL GIF EXPORT
# CLEAN + STABLE + NO ARTIFACTS
# ══════════════════════════════════════════════════════════════════════════

import os
import cv2
import imageio
import numpy as np

from IPython.display import Image, display


# ─────────────────────────────────────────────────────────
# USE THE BEST PREDICTIONS
# (NOT overprocessed versions)
# ─────────────────────────────────────────────────────────

conv_preds = preds_convlstm_v3
spatial_preds = preds_spatial_v3


# ─────────────────────────────────────────────────────────
# SAVE DIRECTORY
# ─────────────────────────────────────────────────────────

EXPORT_DIR = "./professional_gifs"

os.makedirs(
    EXPORT_DIR,
    exist_ok=True
)


# ─────────────────────────────────────────────────────────
# CLEAN UPSCALER
# ─────────────────────────────────────────────────────────

def upscale_smooth(
    frame,
    scale=6
):

    frame = (
        frame.numpy() * 255
    ).clip(0,255).astype(np.uint8)

    # BEST interpolation for grayscale video
    frame = cv2.resize(
        frame,
        (64*scale, 64*scale),
        interpolation=cv2.INTER_AREA
    )

    return frame


# ─────────────────────────────────────────────────────────
# CLEAN TITLE BAR
# ─────────────────────────────────────────────────────────

def add_title(
    img,
    text
):

    canvas = cv2.copyMakeBorder(
        img,
        38,
        0,
        0,
        0,
        cv2.BORDER_CONSTANT,
        value=255
    )

    cv2.putText(
        canvas,
        text,
        (15,26),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0,),
        2,
        cv2.LINE_AA
    )

    return canvas


# ─────────────────────────────────────────────────────────
# FRAME INTERPOLATION
# Makes GIF motion smoother
# ─────────────────────────────────────────────────────────

def interpolate_frames(
    f1,
    f2,
    n=2
):

    out = []

    for i in range(n):

        alpha = i / n

        blended = (
            f1 * (1 - alpha)
            +
            f2 * alpha
        )

        out.append(
            blended.astype(np.uint8)
        )

    return out


# ─────────────────────────────────────────────────────────
# PROFESSIONAL GIF EXPORT
# ─────────────────────────────────────────────────────────

def export_professional_gif(
    observed,
    predicted,
    gt,
    save_path,
    model_name="Model",
    fps=8
):

    frames = []

    # ─────────────────────────────────────
    # OBSERVED CONTEXT
    # ─────────────────────────────────────

    for t in range(observed.shape[0]):

        obs = upscale_smooth(
            observed[t,0]
        )

        panel = np.concatenate(
            [
                obs,
                obs
            ],
            axis=1
        )

        panel = add_title(
            panel,
            f"Observed Context | Frame {t+1}"
        )

        frames.append(panel)

    # ─────────────────────────────────────
    # FUTURE PREDICTIONS
    # ─────────────────────────────────────

    pred_frames = []

    for t in range(predicted.shape[0]):

        gt_f = upscale_smooth(
            gt[t,0]
        )

        pred_f = upscale_smooth(
            predicted[t,0]
        )

        combined = np.concatenate(
            [
                gt_f,
                pred_f
            ],
            axis=1
        )

        combined = add_title(
            combined,
            f"{model_name} | Ground Truth vs Prediction | Step {t+1}"
        )

        pred_frames.append(combined)

    # ─────────────────────────────────────
    # INTERPOLATE FOR SMOOTHNESS
    # ─────────────────────────────────────

    smooth_frames = []

    for i in range(len(pred_frames)-1):

        smooth_frames.append(
            pred_frames[i]
        )

        inter = interpolate_frames(
            pred_frames[i],
            pred_frames[i+1],
            n=2
        )

        smooth_frames.extend(inter)

    smooth_frames.append(
        pred_frames[-1]
    )

    frames.extend(smooth_frames)

    # ─────────────────────────────────────
    # SAVE
    # ─────────────────────────────────────

    imageio.mimsave(
        save_path,
        frames,
        fps=fps
    )

    print(f"✓ Saved → {save_path}")


# ─────────────────────────────────────────────────────────
# EXPORT CONVLSTM
# ─────────────────────────────────────────────────────────

export_professional_gif(
    input_seq[0].cpu(),
    conv_preds[0],
    target_seq[0],
    f"{EXPORT_DIR}/professional_convlstm.gif",
    model_name="ConvLSTM"
)


# ─────────────────────────────────────────────────────────
# EXPORT SPATIAL
# ─────────────────────────────────────────────────────────

export_professional_gif(
    input_seq[0].cpu(),
    spatial_preds[0],
    target_seq[0],
    f"{EXPORT_DIR}/professional_spatial.gif",
    model_name="Spatial"
)

print("\n✓ PROFESSIONAL GIF EXPORT COMPLETE")


# ─────────────────────────────────────────────────────────
# DISPLAY
# ─────────────────────────────────────────────────────────

display(
    Image(
        filename=f"{EXPORT_DIR}/professional_convlstm.gif"
    )
)

display(
    Image(
        filename=f"{EXPORT_DIR}/professional_spatial.gif"
    )
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# AI DEMO STYLE FUTURE MOTION GIF
# CONTEXT → FUTURE PREDICTION
# ══════════════════════════════════════════════════════════════════════════

import os
import cv2
import imageio
import numpy as np

from IPython.display import Image, display


# ─────────────────────────────────────────────────────────
# SAVE DIR
# ─────────────────────────────────────────────────────────

DEMO_DIR = "./ai_demo_motion"

os.makedirs(
    DEMO_DIR,
    exist_ok=True
)


# ─────────────────────────────────────────────────────────
# USE BEST PREDICTIONS
# ─────────────────────────────────────────────────────────

demo_preds = preds_convlstm_v3


# ─────────────────────────────────────────────────────────
# UPSCALE
# ─────────────────────────────────────────────────────────

def upscale(frame, scale=7):

    frame = (
        frame.numpy() * 255
    ).clip(0,255).astype(np.uint8)

    frame = cv2.resize(
        frame,
        (64*scale, 64*scale),
        interpolation=cv2.INTER_CUBIC
    )

    return frame


# ─────────────────────────────────────────────────────────
# ADD MODERN TITLE
# ─────────────────────────────────────────────────────────

def add_title(img, text):

    canvas = cv2.copyMakeBorder(
        img,
        50,
        20,
        20,
        20,
        cv2.BORDER_CONSTANT,
        value=255
    )

    cv2.putText(
        canvas,
        text,
        (20,35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0,),
        2,
        cv2.LINE_AA
    )

    return canvas


# ─────────────────────────────────────────────────────────
# CREATE DEMO GIF
# ─────────────────────────────────────────────────────────

frames = []

# OBSERVED CONTEXT
for t in range(input_seq.shape[1]):

    frame = upscale(
        input_seq[0,t,0].cpu()
    )

    frame = add_title(
        frame,
        f"Observed Context Frame {t+1}"
    )

    frames.append(frame)

# PAUSE BEFORE PREDICTION
for _ in range(6):

    frames.append(frames[-1])

# FUTURE PREDICTIONS
for t in range(demo_preds.shape[1]):

    frame = upscale(
        demo_preds[0,t,0]
    )

    frame = add_title(
        frame,
        f"AI Predicted Future Motion | Step {t+1}"
    )

    frames.append(frame)

# HOLD FINAL FRAME
for _ in range(8):

    frames.append(frames[-1])

# SAVE
SAVE_PATH = f"{DEMO_DIR}/future_motion_demo.gif"

imageio.mimsave(
    SAVE_PATH,
    frames,
    fps=5
)

print(f"✓ Saved → {SAVE_PATH}")

# DISPLAY
display(Image(filename=SAVE_PATH))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL CINEMATIC MP4 EXPORT
# BEST QUALITY OUTPUT
# ══════════════════════════════════════════════════════════════════════════

import os
import cv2
import numpy as np


# ─────────────────────────────────────────────────────────
# SAVE DIR
# ─────────────────────────────────────────────────────────

VIDEO_DIR = "./final_video_demo"

os.makedirs(
    VIDEO_DIR,
    exist_ok=True
)


# ─────────────────────────────────────────────────────────
# USE BEST PREDICTIONS
# ─────────────────────────────────────────────────────────

demo_preds = preds_convlstm_v3


# ─────────────────────────────────────────────────────────
# UPSCALE
# ─────────────────────────────────────────────────────────

def upscale(frame, scale=8):

    frame = (
        frame.numpy() * 255
    ).clip(0,255).astype(np.uint8)

    frame = cv2.resize(
        frame,
        (64*scale, 64*scale),
        interpolation=cv2.INTER_CUBIC
    )

    return frame


# ─────────────────────────────────────────────────────────
# TITLE
# ─────────────────────────────────────────────────────────

def add_title(img, text):

    canvas = cv2.copyMakeBorder(
        img,
        60,
        20,
        20,
        20,
        cv2.BORDER_CONSTANT,
        value=255
    )

    cv2.putText(
        canvas,
        text,
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0,),
        2,
        cv2.LINE_AA
    )

    return canvas


# ─────────────────────────────────────────────────────────
# VIDEO SETTINGS
# ─────────────────────────────────────────────────────────

SAVE_PATH = f"{VIDEO_DIR}/future_motion_demo.mp4"

fps = 10

sample = upscale(
    input_seq[0,0,0].cpu()
)

sample = add_title(
    sample,
    "Sample"
)

height, width = sample.shape

writer = cv2.VideoWriter(
    SAVE_PATH,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height),
    isColor=False
)


# ─────────────────────────────────────────────────────────
# OBSERVED FRAMES
# ─────────────────────────────────────────────────────────

for t in range(input_seq.shape[1]):

    frame = upscale(
        input_seq[0,t,0].cpu()
    )

    frame = add_title(
        frame,
        f"Observed Context Frame {t+1}"
    )

    writer.write(frame)


# HOLD
for _ in range(8):

    writer.write(frame)


# ─────────────────────────────────────────────────────────
# PREDICTED FUTURE
# ─────────────────────────────────────────────────────────

for t in range(demo_preds.shape[1]):

    frame = upscale(
        demo_preds[0,t,0]
    )

    frame = add_title(
        frame,
        f"AI Predicted Future Motion | Step {t+1}"
    )

    writer.write(frame)


# HOLD FINAL
for _ in range(12):

    writer.write(frame)

writer.release()

print(f"✓ Saved MP4 → {SAVE_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# GIF GENERATION
# ══════════════════════════════════════════════════════════════════════════
def to_u8(tensor):
    return [(tensor[t,0].numpy()*255).clip(0,255).astype(np.uint8)
            for t in range(tensor.shape[0])]


def make_panel(frame_u8, label, color, scale=3):
    from PIL import ImageDraw
    img = Image.fromarray(frame_u8).resize((64*scale, 64*scale), Image.LANCZOS)
    arr = np.array(img)
    rgb = np.stack([arr]*3, axis=-1)
    bar = np.zeros((18, rgb.shape[1], 3), dtype=np.uint8)
    bar[0:3] = color
    full = np.vstack([bar, rgb])
    img2 = Image.fromarray(full)
    draw = ImageDraw.Draw(img2)
    draw.text((4, 3), label, fill=tuple(color))
    return np.array(img2)


def make_comparison_gif(observed, pred_conv, pred_spatial, ground_truth,
                         save_path, fps=5, scale=3):
    """4-panel GIF. Observation phase plays at fps//2 so viewer sees the person."""
    colors = {
        'Observed':   (100, 149, 237),
        'ConvLSTM':   ( 60, 179, 113),
        'Spatial TX': (255, 165,   0),
        'GT Future':  (220,  20,  60),
    }
    obs_f  = to_u8(observed)
    conv_f = to_u8(pred_conv)
    spat_f = to_u8(pred_spatial)
    gt_f   = to_u8(ground_truth)
    div    = np.ones((64*scale+18, 3, 3), dtype=np.uint8) * 50

    obs_frames  = []
    pred_frames = []

    for t in range(len(obs_f)):
        panels = [
            make_panel(obs_f[t],  'Observed',   colors['Observed'],   scale),
            make_panel(obs_f[t],  'ConvLSTM',   colors['ConvLSTM'],   scale),
            make_panel(obs_f[t],  'Spatial TX', colors['Spatial TX'], scale),
            make_panel(obs_f[t],  'GT Future',  colors['GT Future'],  scale),
        ]
        from PIL import ImageDraw
        row  = np.concatenate([panels[0],div,panels[1],div,panels[2],div,panels[3]], axis=1)
        img  = Image.fromarray(row)
        draw = ImageDraw.Draw(img)
        draw.text((4, row.shape[0]-14), f't={t+1}/{len(obs_f)+len(conv_f)}  [OBSERVING]',
                  fill=(200,200,200))
        obs_frames.append(np.array(img))

    for t in range(len(conv_f)):
        panels = [
            make_panel(obs_f[-1],  'Observed',   colors['Observed'],   scale),
            make_panel(conv_f[t],  'ConvLSTM',   colors['ConvLSTM'],   scale),
            make_panel(spat_f[t],  'Spatial TX', colors['Spatial TX'], scale),
            make_panel(gt_f[t],    'GT Future',  colors['GT Future'],  scale),
        ]
        from PIL import ImageDraw
        row  = np.concatenate([panels[0],div,panels[1],div,panels[2],div,panels[3]], axis=1)
        img  = Image.fromarray(row)
        draw = ImageDraw.Draw(img)
        draw.text((4, row.shape[0]-14), f't={len(obs_f)+t+1}/{len(obs_f)+len(conv_f)}  [PREDICTING]',
                  fill=(200,200,200))
        pred_frames.append(np.array(img))

    # Observation plays at half speed so viewer registers the person
    slow_obs = []
    for f in obs_frames:
        slow_obs.append(f); slow_obs.append(f)   # each obs frame shown twice

    imageio.mimsave(save_path, slow_obs + pred_frames, fps=fps, loop=0)
    print(f'Saved → {save_path}  ({os.path.getsize(save_path)/1e6:.1f}MB)')


def make_single_gif(observed, predicted, label, border_color, save_path, fps=6, scale=3):
    obs_f  = to_u8(observed)
    pred_f = to_u8(predicted)
    n_obs  = len(obs_f)

    obs_gif = []; pred_gif = []
    for t, frame in enumerate(obs_f):
        from PIL import ImageDraw
        img = Image.fromarray(frame).resize((64*scale, 64*scale), Image.LANCZOS)
        arr = np.stack([np.array(img)]*3, axis=-1)
        bc  = (100,149,237)
        arr[0:5]=arr[-5:]=arr[:,0:5]=arr[:,-5:]=bc
        img2 = Image.fromarray(arr.astype(np.uint8))
        draw = ImageDraw.Draw(img2)
        draw.text((6,6), f'OBSERVED t={t+1}', fill=bc)
        obs_gif.append(np.array(img2))

    for t, frame in enumerate(pred_f):
        from PIL import ImageDraw
        img = Image.fromarray(frame).resize((64*scale, 64*scale), Image.LANCZOS)
        arr = np.stack([np.array(img)]*3, axis=-1)
        bc  = border_color
        arr[0:5]=arr[-5:]=arr[:,0:5]=arr[:,-5:]=bc
        img2 = Image.fromarray(arr.astype(np.uint8))
        draw = ImageDraw.Draw(img2)
        draw.text((6,6), f'{label} t={n_obs+t+1}', fill=bc)
        pred_gif.append(np.array(img2))

    # slow obs, normal pred
    slow_obs = [f for frame in obs_gif for f in [frame, frame]]
    imageio.mimsave(save_path, slow_obs + pred_gif, fps=fps, loop=0)
    print(f'Saved → {save_path}  ({os.path.getsize(save_path)/1e6:.1f}MB)')


def make_stochastic_gif(observed, f_low, f_mid, f_high, save_path, fps=5, scale=3):
    colors = {
        'Future A σ=0.02': (100,200,100),
        'Future B σ=0.08': (255,165,  0),
        'Future C σ=0.15': (220, 80, 80),
    }
    obs_f  = to_u8(observed)
    low_f  = to_u8(f_low)
    mid_f  = to_u8(f_mid)
    high_f = to_u8(f_high)
    div    = np.ones((64*scale+18, 3, 3), dtype=np.uint8)*50

    obs_gif = []; pred_gif = []

    for t in range(len(obs_f)):
        panels = [
            make_panel(obs_f[t], 'Future A σ=0.02', colors['Future A σ=0.02'], scale),
            make_panel(obs_f[t], 'Future B σ=0.08', colors['Future B σ=0.08'], scale),
            make_panel(obs_f[t], 'Future C σ=0.15', colors['Future C σ=0.15'], scale),
        ]
        from PIL import ImageDraw
        row = np.concatenate([panels[0],div,panels[1],div,panels[2]], axis=1)
        img = Image.fromarray(row); draw = ImageDraw.Draw(img)
        draw.text((4, row.shape[0]-14), f't={t+1}  [OBSERVING]', fill=(200,200,200))
        obs_gif.append(np.array(img))

    for t in range(len(low_f)):
        panels = [
            make_panel(low_f[t],  'Future A σ=0.02', colors['Future A σ=0.02'], scale),
            make_panel(mid_f[t],  'Future B σ=0.08', colors['Future B σ=0.08'], scale),
            make_panel(high_f[t], 'Future C σ=0.15', colors['Future C σ=0.15'], scale),
        ]
        from PIL import ImageDraw
        row = np.concatenate([panels[0],div,panels[1],div,panels[2]], axis=1)
        img = Image.fromarray(row); draw = ImageDraw.Draw(img)
        draw.text((4, row.shape[0]-14), f't={len(obs_f)+t+1}  [PREDICTING]', fill=(200,200,200))
        pred_gif.append(np.array(img))

    slow_obs = [f for frame in obs_gif for f in [frame, frame]]
    imageio.mimsave(save_path, slow_obs + pred_gif, fps=fps, loop=0)
    print(f'Saved → {save_path}  ({os.path.getsize(save_path)/1e6:.1f}MB)')


print('GIF functions ready.')

In [ ]:
# FIX PIL IMAGE OVERRIDE

from PIL import Image as PILImage
from PIL import ImageDraw

# overwrite broken Image reference
Image = PILImage

print("✓ PIL Image fixed")

In [ ]:
# ── Compute SSIM + LPIPS (models already loaded) ─────────────────────────
from skimage.metrics import structural_similarity as sk_ssim
import lpips, torch, numpy as np

lpips_fn = lpips.LPIPS(net='alex').to(device)
lpips_fn.eval()

ssim_conv     = []
ssim_spatial  = []
lpips_conv    = []
lpips_spatial = []

MAX_BATCHES = 20   # remove limit for full eval

vae_v2.eval(); dynamics_model.eval(); spatial_model.eval()

def to_lpips(arr):
    t = torch.tensor(arr).unsqueeze(0).unsqueeze(0)
    return t.repeat(1, 3, 1, 1).to(device) * 2 - 1

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i >= MAX_BATCHES: break
        batch = batch.to(device)
        inp = batch[:, :10]
        gt  = batch[:, 10:10+PRED_STEPS].cpu()

        pc = rollout_clean(vae_v2, dynamics_model, inp, PRED_STEPS)
        ps = rollout_clean(vae_v2, spatial_model,  inp, PRED_STEPS)

        for b in range(batch.shape[0]):
            for t in range(PRED_STEPS):
                gt_np = gt[b, t, 0].numpy()
                pc_np = pc[b, t, 0].numpy()
                ps_np = ps[b, t, 0].numpy()

                ssim_conv.append(sk_ssim(gt_np, pc_np, data_range=1.0))
                ssim_spatial.append(sk_ssim(gt_np, ps_np, data_range=1.0))
                lpips_conv.append(lpips_fn(to_lpips(pc_np), to_lpips(gt_np)).item())
                lpips_spatial.append(lpips_fn(to_lpips(ps_np), to_lpips(gt_np)).item())

        if (i+1) % 5 == 0:
            print(f"  Batch {i+1}/{MAX_BATCHES} done...")

print(f"\nSSIM  — ConvLSTM: {np.mean(ssim_conv):.4f}  Spatial TX: {np.mean(ssim_spatial):.4f}")
print(f"LPIPS — ConvLSTM: {np.mean(lpips_conv):.4f}  Spatial TX: {np.mean(lpips_spatial):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CLEAN SHORT FUTURE ROLLOUT
# BEST VISUAL RESULT
# ══════════════════════════════════════════════════════════════════════════

PRED_STEPS = 6

# ─────────────────────────────────────────
# GENERATE STABLE PREDICTIONS
# ─────────────────────────────────────────

preds_convlstm_short = rollout_stable_v3(
    vae_v2,
    dynamics_model,
    input_seq,
    PRED_STEPS
)

preds_spatial_short = rollout_stable_v3(
    vae_v2,
    spatial_model,
    input_seq,
    PRED_STEPS
)

print("✓ Short rollout predictions generated")


# ─────────────────────────────────────────
# TARGET
# ─────────────────────────────────────────

target_seq_short = target_seq[:, :PRED_STEPS]


# ─────────────────────────────────────────
# EXPORT CLEAN GIF
# ─────────────────────────────────────────

SAVE_PATH = "./BEST_SHORT_DEMO"

os.makedirs(
    SAVE_PATH,
    exist_ok=True
)

make_comparison_gif(
    input_seq[0].cpu(),
    preds_convlstm_short[0],
    preds_spatial_short[0],
    target_seq_short[0],
    f"{SAVE_PATH}/short_future_prediction.gif",
    fps=5,
    scale=4
)

print("\n✓ FINAL SHORT DEMO GIF GENERATED")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  RESEARCH QUALITY VISUALIZATION SUITE
# FINAL PROFESSIONAL CHARTS
# ══════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────
# SAVE DIRECTORY
# ─────────────────────────────────────────────────────────

GRAPH_DIR = "./graphs"

os.makedirs(
    GRAPH_DIR,
    exist_ok=True
)

# ─────────────────────────────────────────────────────────
# STYLE
# ─────────────────────────────────────────────────────────

plt.style.use('seaborn-v0_8-whitegrid')

TITLE_SIZE = 20
SUBTITLE_SIZE = 13
LABEL_SIZE = 14
TICK_SIZE = 12

# COLORS
vae_color = '#1E88E5'
conv_color = '#43A047'
spatial_color = '#FB8C00'

# ─────────────────────────────────────────────────────────
# FIGURE 1 — TRAINING CURVES
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18,5)
)

fig.suptitle(
    'Latent World Model Training Dynamics',
    fontsize=22,
    fontweight='bold',
    y=1.02
)

# ─────────────────────────────────────
# VAE
# ─────────────────────────────────────

ax = axes[0]

epochs = np.arange(len(vae_v2_losses))

ax.plot(
    epochs,
    vae_v2_losses,
    color=vae_color,
    linewidth=2.5
)

ax.fill_between(
    epochs,
    vae_v2_losses,
    alpha=0.15,
    color=vae_color
)

ax.set_title(
    f'VAE v2 ({len(vae_v2_losses)} epochs)',
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel(
    'Epoch',
    fontsize=LABEL_SIZE
)

ax.set_ylabel(
    'Reconstruction Loss',
    fontsize=LABEL_SIZE
)

ax.tick_params(labelsize=TICK_SIZE)

ax.text(
    len(vae_v2_losses)-1,
    vae_v2_losses[-1],
    f'Final: {vae_v2_losses[-1]:.4f}',
    fontsize=10,
    color=vae_color
)

# ─────────────────────────────────────
# CONVLSTM
# ─────────────────────────────────────

ax = axes[1]

epochs = np.arange(len(dynamics_losses))

ax.plot(
    epochs,
    dynamics_losses,
    color=conv_color,
    linewidth=2.5
)

ax.fill_between(
    epochs,
    dynamics_losses,
    alpha=0.15,
    color=conv_color
)

ax.set_title(
    f'ConvLSTM ({len(dynamics_losses)} epochs)',
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel(
    'Epoch',
    fontsize=LABEL_SIZE
)

ax.set_ylabel(
    'Latent Prediction Loss',
    fontsize=LABEL_SIZE
)

ax.tick_params(labelsize=TICK_SIZE)

ax.text(
    len(dynamics_losses)-1,
    dynamics_losses[-1],
    f'Final: {dynamics_losses[-1]:.4f}',
    fontsize=10,
    color=conv_color
)

# ─────────────────────────────────────
# SPATIAL TX
# ─────────────────────────────────────

ax = axes[2]

epochs = np.arange(len(spatial_losses))

ax.plot(
    epochs,
    spatial_losses,
    color=spatial_color,
    linewidth=2.5
)

ax.fill_between(
    epochs,
    spatial_losses,
    alpha=0.15,
    color=spatial_color
)

ax.set_title(
    f'Spatial Transformer ({len(spatial_losses)} epochs)',
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel(
    'Epoch',
    fontsize=LABEL_SIZE
)

ax.set_ylabel(
    'Latent Prediction Loss',
    fontsize=LABEL_SIZE
)

ax.tick_params(labelsize=TICK_SIZE)

ax.text(
    len(spatial_losses)-1,
    spatial_losses[-1],
    f'Final: {spatial_losses[-1]:.4f}',
    fontsize=10,
    color=spatial_color
)

plt.tight_layout()

SAVE_PATH = f"{GRAPH_DIR}/training_curves.png"

plt.savefig(
    SAVE_PATH,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print(f"✓ Saved → {SAVE_PATH}")


# ─────────────────────────────────────────────────────────
# FIGURE 2 — SSIM + LPIPS COMPARISON
# ─────────────────────────────────────────────────────────

fig = plt.figure(figsize=(10,6))

models = ['ConvLSTM', 'Spatial TX']

ssim_vals = [
    np.mean(ssim_conv),
    np.mean(ssim_spatial)
]

lpips_vals = [
    np.mean(lpips_conv),
    np.mean(lpips_spatial)
]

x = np.arange(len(models))

width = 0.35

plt.bar(
    x - width/2,
    ssim_vals,
    width,
    label='SSIM ↑',
    color=conv_color
)

plt.bar(
    x + width/2,
    lpips_vals,
    width,
    label='LPIPS ↓',
    color=spatial_color
)

# value labels
for i,v in enumerate(ssim_vals):

    plt.text(
        i - width/2,
        v + 0.01,
        f'{v:.3f}',
        ha='center',
        fontsize=12
    )

for i,v in enumerate(lpips_vals):

    plt.text(
        i + width/2,
        v + 0.01,
        f'{v:.3f}',
        ha='center',
        fontsize=12
    )

plt.xticks(
    x,
    models,
    fontsize=TICK_SIZE
)

plt.yticks(fontsize=TICK_SIZE)

plt.ylabel(
    'Metric Score',
    fontsize=LABEL_SIZE
)

plt.title(
    'Future Frame Prediction Performance',
    fontsize=TITLE_SIZE,
    fontweight='bold',
    pad=20
)

plt.legend(fontsize=12)

plt.grid(alpha=0.25)

plt.tight_layout()

SAVE_PATH = f"{GRAPH_DIR}/metrics_comparison.png"

plt.savefig(
    SAVE_PATH,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print(f"✓ Saved → {SAVE_PATH}")


# ─────────────────────────────────────────────────────────
# FIGURE 3 — TEMPORAL STABILITY
# ─────────────────────────────────────────────────────────

conv_curve = []
spatial_curve = []

for t in range(PRED_STEPS):

    conv_t = ssim_conv[t::PRED_STEPS]

    spatial_t = ssim_spatial[t::PRED_STEPS]

    conv_curve.append(
        np.mean(conv_t)
    )

    spatial_curve.append(
        np.mean(spatial_t)
    )

steps = np.arange(1, PRED_STEPS+1)

fig = plt.figure(figsize=(10,6))

plt.plot(
    steps,
    conv_curve,
    marker='o',
    markersize=10,
    linewidth=3,
    color=conv_color,
    label='ConvLSTM'
)

plt.plot(
    steps,
    spatial_curve,
    marker='o',
    markersize=10,
    linewidth=3,
    color=spatial_color,
    label='Spatial TX'
)

# annotations
for x,y in zip(steps, conv_curve):

    plt.text(
        x,
        y + 0.001,
        f'{y:.3f}',
        ha='center',
        fontsize=11
    )

for x,y in zip(steps, spatial_curve):

    plt.text(
        x,
        y - 0.003,
        f'{y:.3f}',
        ha='center',
        fontsize=11
    )

plt.xlabel(
    'Prediction Horizon',
    fontsize=LABEL_SIZE
)

plt.ylabel(
    'SSIM',
    fontsize=LABEL_SIZE
)

plt.title(
    'Temporal Prediction Stability',
    fontsize=TITLE_SIZE,
    fontweight='bold',
    pad=20
)

plt.xticks(
    steps,
    fontsize=TICK_SIZE
)

plt.yticks(fontsize=TICK_SIZE)

plt.legend(fontsize=12)

plt.grid(alpha=0.25)

plt.tight_layout()

SAVE_PATH = f"{GRAPH_DIR}/temporal_stability.png"

plt.savefig(
    SAVE_PATH,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print(f"✓ Saved → {SAVE_PATH}")


print("\n✓ ALL PROFESSIONAL GRAPHS GENERATED")
print(f"✓ Saved inside: {GRAPH_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SAVE + ZIP ALL GENERATED GIFS
# ══════════════════════════════════════════════════════════════════════════

import os
import shutil
import zipfile

# ─────────────────────────────────────────────────────────
# COLLECT ALL GIF DIRECTORIES
# ─────────────────────────────────────────────────────────

gif_dirs = [
    "./FINAL_DEMO_GIFS",
    "./BEST_SHORT_DEMO",
    "./professional_gifs",
    "./final_clean_gifs",
    "./final_cinematic_gifs",
    "./ai_demo_motion"
]

# ─────────────────────────────────────────────────────────
# OUTPUT DIRECTORY
# ─────────────────────────────────────────────────────────

MASTER_DIR = "./ALL_PROJECT_OUTPUTS"

os.makedirs(
    MASTER_DIR,
    exist_ok=True
)

# ─────────────────────────────────────────────────────────
# COPY ALL GIFS
# ─────────────────────────────────────────────────────────

copied = 0

for gdir in gif_dirs:

    if os.path.exists(gdir):

        for file in os.listdir(gdir):

            if file.endswith(".gif"):

                src = os.path.join(gdir, file)

                dst = os.path.join(
                    MASTER_DIR,
                    file
                )

                shutil.copy(
                    src,
                    dst
                )

                copied += 1

print(f"✓ Copied {copied} GIFs")


# ─────────────────────────────────────────────────────────
# ALSO COPY MP4s
# ─────────────────────────────────────────────────────────

video_dirs = [
    "./final_video_demo"
]

for vdir in video_dirs:

    if os.path.exists(vdir):

        for file in os.listdir(vdir):

            if file.endswith(".mp4"):

                src = os.path.join(vdir, file)

                dst = os.path.join(
                    MASTER_DIR,
                    file
                )

                shutil.copy(
                    src,
                    dst
                )

                print(f"✓ Copied video: {file}")


# ─────────────────────────────────────────────────────────
# ZIP EVERYTHING
# ─────────────────────────────────────────────────────────

ZIP_PATH = "./world_model_outputs.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    'w',
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(MASTER_DIR):

        for file in files:

            path = os.path.join(root, file)

            zipf.write(
                path,
                arcname=file
            )

print(f"\n✓ ZIP CREATED → {ZIP_PATH}")


# ─────────────────────────────────────────────────────────
# SHOW SAVED FILES
# ─────────────────────────────────────────────────────────

print("\nSaved files:")

for file in os.listdir(MASTER_DIR):

    print(" •", file)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# UPLOAD ALL RESULTS TO HUGGING FACE
# MODELS + GIFS + VIDEOS + GRAPHS
# ══════════════════════════════════════════════════════════════════════════

!pip -q install huggingface_hub

import os
from huggingface_hub import login
from huggingface_hub import HfApi


# ─────────────────────────────────────────────────────────
# HUGGING FACE CONFIG
# ─────────────────────────────────────────────────────────

HF_TOKEN   = #entered
HF_REPO_ID = #entered
login(token=HF_TOKEN)

api = HfApi()

print("✓ Logged into Hugging Face")


# ─────────────────────────────────────────────────────────
# FILES TO UPLOAD
# ─────────────────────────────────────────────────────────

upload_files = []

# model files
model_files = [
    "vae_v2.pth",
    "dynamics_v2.pth",
    "spatial_transformer.pth"
]

for f in model_files:

    if os.path.exists(f):

        upload_files.append(f)


# GIF directories
gif_dirs = [
    "./FINAL_DEMO_GIFS",
    "./BEST_SHORT_DEMO",
    "./professional_gifs",
    "./final_clean_gifs",
    "./final_cinematic_gifs",
    "./ai_demo_motion"
]

for gdir in gif_dirs:

    if os.path.exists(gdir):

        for file in os.listdir(gdir):

            path = os.path.join(gdir, file)

            upload_files.append(path)


# video dirs
video_dirs = [
    "./final_video_demo"
]

for vdir in video_dirs:

    if os.path.exists(vdir):

        for file in os.listdir(vdir):

            path = os.path.join(vdir, file)

            upload_files.append(path)


# graph dirs
graph_dirs = [
    "./graphs"
]

for gdir in graph_dirs:

    if os.path.exists(gdir):

        for file in os.listdir(gdir):

            path = os.path.join(gdir, file)

            upload_files.append(path)


# zip
if os.path.exists("./world_model_outputs.zip"):

    upload_files.append("./world_model_outputs.zip")


print(f"\nFound {len(upload_files)} files to upload")


# ─────────────────────────────────────────────────────────
# UPLOAD
# ─────────────────────────────────────────────────────────

for file_path in upload_files:

    try:

        repo_path = os.path.basename(file_path)

        print(f"\nUploading: {repo_path}")

        api.upload_file(
            path_or_fileobj=file_path,
            path_in_repo=repo_path,
            repo_id=HF_REPO_ID,
            repo_type="model"
        )

        print("✓ Uploaded")

    except Exception as e:

        print(f"✗ Failed: {e}")


# ─────────────────────────────────────────────────────────
# DONE
# ─────────────────────────────────────────────────────────

print("\n══════════════════════════════════════")
print("✓ ALL RESULTS UPLOADED TO HUGGING FACE")
print(f"✓ Repo: https://huggingface.co/SimranjitKaur/vae_kth-world-model")
print("══════════════════════════════════════")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SAVE + UPLOAD PIPELINE VISUALIZATION IMAGES
# ══════════════════════════════════════════════════════════════════════════

import os
import matplotlib.pyplot as plt

from huggingface_hub import HfApi


# ─────────────────────────────────────────────────────────
# SAVE DIR
# ─────────────────────────────────────────────────────────

PIPELINE_DIR = "./pipeline_images"

os.makedirs(
    PIPELINE_DIR,
    exist_ok=True
)


# ─────────────────────────────────────────────────────────
# SAVE ALL OPEN FIGURES
# ─────────────────────────────────────────────────────────

fig_nums = plt.get_fignums()

print(f"Found {len(fig_nums)} matplotlib figures")

saved_files = []

for i, fig_num in enumerate(fig_nums):

    fig = plt.figure(fig_num)

    save_path = os.path.join(
        PIPELINE_DIR,
        f"pipeline_figure_{i+1}.png"
    )

    fig.savefig(
        save_path,
        dpi=300,
        bbox_inches='tight'
    )

    saved_files.append(save_path)

    print(f"✓ Saved → {save_path}")


# ─────────────────────────────────────────────────────────
# UPLOAD TO HUGGING FACE
# ─────────────────────────────────────────────────────────

api = HfApi()

for img_path in saved_files:

    try:

        filename = os.path.basename(img_path)

        print(f"\nUploading: {filename}")

        api.upload_file(
            path_or_fileobj=img_path,
            path_in_repo=f"pipeline_images/{filename}",
            repo_id=HF_REPO_ID,
            repo_type="model"
        )

        print("✓ Uploaded")

    except Exception as e:

        print(f"✗ Failed: {e}")


print("\n══════════════════════════════════════")
print("✓ ALL PIPELINE IMAGES UPLOADED")
print(f"✓ Repo: https://huggingface.co/{HF_REPO_ID}")
print("══════════════════════════════════════")